# PySpark CDC — Streaming Micro-batch Simulation

Structured Streaming style CDC processing without external storage. Each Python list below simulates one streaming micro-batch.

## 00 — Spark setup and sample data

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("cdc-streaming-foreachbatch")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/cdc-streaming-foreachbatch_warehouse")
    .getOrCreate()
)

spark.version

'4.0.2'

In [2]:
current_state = spark.createDataFrame([
    (1, "Alice", "alice@old.com", "2026-05-02T09:00:00"),
    (2, "Bob", "bob@mail.com", "2026-05-02T09:00:00"),
], ["customer_id", "name", "email", "updated_at"]).withColumn(
    "updated_at",
    F.to_timestamp("updated_at")
)

micro_batches = [
    [
        (1, "Alice", "alice@new.com", "U", "2026-05-02T10:05:00"),
        (3, "Carol", "carol@mail.com", "I", "2026-05-02T10:10:00"),
    ],
    [
        (2, None, None, "D", "2026-05-02T10:20:00"),
        (3, "Carol", "carol.new@mail.com", "U", "2026-05-02T10:25:00"),
    ],
]

current_state.orderBy("customer_id").show(truncate=False)

[Stage 0:>                                                          (0 + 2) / 2]

+-----------+-----+-------------+-------------------+
|customer_id|name |email        |updated_at         |
+-----------+-----+-------------+-------------------+
|1          |Alice|alice@old.com|2026-05-02 09:00:00|
|2          |Bob  |bob@mail.com |2026-05-02 09:00:00|
+-----------+-----+-------------+-------------------+



## 01 — Reusable CDC apply function

In [3]:
from pyspark.sql.window import Window

def apply_cdc_scd1(current_df, cdc_df):
    latest_window = Window.partitionBy("customer_id").orderBy(F.col("event_time").desc())

    latest = (
        cdc_df
        .withColumn("rn", F.row_number().over(latest_window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    delete_keys = latest.filter(F.col("op") == "D").select("customer_id")

    after_deletes = current_df.join(delete_keys, on="customer_id", how="left_anti")

    upserts = (
        latest
        .filter(F.col("op").isin("I", "U"))
        .select(
            "customer_id",
            "name",
            "email",
            F.col("event_time").alias("updated_at")
        )
    )

    unchanged = after_deletes.join(
        upserts.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )

    return unchanged.unionByName(upserts)

## 02 — Process simulated micro-batches

In [4]:
schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("op", StringType(), False),
    StructField("event_time", StringType(), False),
])

state = current_state

for batch_id, rows in enumerate(micro_batches, start=1):
    print(f"Micro-batch {batch_id}")

    batch_df = spark.createDataFrame(rows, schema).withColumn(
        "event_time",
        F.to_timestamp("event_time")
    )

    print("CDC events")
    batch_df.orderBy("event_time").show(truncate=False)

    state = apply_cdc_scd1(state, batch_df)

    print("Current state after micro-batch")
    state.orderBy("customer_id").show(truncate=False)

Micro-batch 1
CDC events
+-----------+-----+--------------+---+-------------------+
|customer_id|name |email         |op |event_time         |
+-----------+-----+--------------+---+-------------------+
|1          |Alice|alice@new.com |U  |2026-05-02 10:05:00|
|3          |Carol|carol@mail.com|I  |2026-05-02 10:10:00|
+-----------+-----+--------------+---+-------------------+

Current state after micro-batch


+-----------+-----+--------------+-------------------+
|customer_id|name |email         |updated_at         |
+-----------+-----+--------------+-------------------+
|1          |Alice|alice@new.com |2026-05-02 10:05:00|
|2          |Bob  |bob@mail.com  |2026-05-02 09:00:00|
|3          |Carol|carol@mail.com|2026-05-02 10:10:00|
+-----------+-----+--------------+-------------------+

Micro-batch 2
CDC events
+-----------+-----+------------------+---+-------------------+
|customer_id|name |email             |op |event_time         |
+-----------+-----+------------------+---+-------------------+
|2          |NULL |NULL              |D  |2026-05-02 10:20:00|
|3          |Carol|carol.new@mail.com|U  |2026-05-02 10:25:00|
+-----------+-----+------------------+---+-------------------+

Current state after micro-batch


+-----------+-----+------------------+-------------------+
|customer_id|name |email             |updated_at         |
+-----------+-----+------------------+-------------------+
|1          |Alice|alice@new.com     |2026-05-02 10:05:00|
|3          |Carol|carol.new@mail.com|2026-05-02 10:25:00|
+-----------+-----+------------------+-------------------+

